<a href="https://colab.research.google.com/github/sairas2124/Gradient_descent-/blob/main/Audio_RAVDESS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install kaggle librosa soundfile -q

In [ ]:
import kagglehub
import os

dataset_path = kagglehub.dataset_download(
    "uwrfkaggler/ravdess-emotional-speech-audio"
)

print("Dataset path:", dataset_path)
print("Folders:", os.listdir(dataset_path))

Using Colab cache for faster access to the 'ravdess-emotional-speech-audio' dataset.
Dataset path: /kaggle/input/ravdess-emotional-speech-audio
Folders: ['Actor_02', 'Actor_17', 'Actor_05', 'Actor_16', 'Actor_21', 'Actor_01', 'Actor_11', 'Actor_20', 'Actor_08', 'Actor_15', 'Actor_06', 'Actor_12', 'Actor_23', 'Actor_24', 'Actor_22', 'Actor_04', 'Actor_19', 'Actor_10', 'Actor_09', 'audio_speech_actors_01-24', 'Actor_14', 'Actor_03', 'Actor_13', 'Actor_18', 'Actor_07']


In [ ]:
for root, dirs, files in os.walk(dataset_path):
    print("Current path:", root)
    print("Number of files:", len(files))
    print("-" * 30)

Current path: /kaggle/input/ravdess-emotional-speech-audio
Number of files: 0
------------------------------
Current path: /kaggle/input/ravdess-emotional-speech-audio/Actor_02
Number of files: 60
------------------------------
Current path: /kaggle/input/ravdess-emotional-speech-audio/Actor_17
Number of files: 60
------------------------------
Current path: /kaggle/input/ravdess-emotional-speech-audio/Actor_05
Number of files: 60
------------------------------
Current path: /kaggle/input/ravdess-emotional-speech-audio/Actor_16
Number of files: 60
------------------------------
Current path: /kaggle/input/ravdess-emotional-speech-audio/Actor_21
Number of files: 60
------------------------------
Current path: /kaggle/input/ravdess-emotional-speech-audio/Actor_01
Number of files: 60
------------------------------
Current path: /kaggle/input/ravdess-emotional-speech-audio/Actor_11
Number of files: 60
------------------------------
Current path: /kaggle/input/ravdess-emotional-speech-audio

In [ ]:
# =========================
# COLLECT FILES + LABELS
# =========================
audio_files = []
labels = []

emotion_map = {
    "01": "neutral",
    "03": "happy",
    "04": "sad",
    "05": "angry"
}

for root, dirs, files in os.walk(dataset_path):
    for file in files:
        if file.endswith(".wav"):
            parts = file.split("-")
            emotion_code = parts[2]

            if emotion_code in emotion_map:
                audio_files.append(os.path.join(root, file))
                labels.append(emotion_map[emotion_code])

print("Total:", len(audio_files))

Total: 1344


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
encoded_labels = le.fit_transform(labels)

print(le.classes_)

['angry' 'happy' 'neutral' 'sad']


In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

if device == "cuda":
    print(torch.cuda.get_device_name(0))

Using: cuda
Tesla T4


In [ ]:
# =========================
# FEATURE EXTRACTION (FIXED SIZE 🔥)
# =========================
import librosa
import numpy as np

def extract_feature(file_path, max_len=128):
    y, sr = librosa.load(file_path, sr=22050)

    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    mel_db = librosa.power_to_db(mel, ref=np.max)

    # FIX SIZE
    if mel_db.shape[1] < max_len:
        pad = max_len - mel_db.shape[1]
        mel_db = np.pad(mel_db, ((0,0),(0,pad)))
    else:
        mel_db = mel_db[:, :max_len]

    return mel_db

In [ ]:
# =========================
# BUILD DATASET
# =========================
X = []

for f in audio_files:
    X.append(extract_feature(f))

X = np.array(X)
X = np.expand_dims(X, axis=1)

print("Shape:", X.shape)   # should be (N,1,128,128)

Shape: (1344, 1, 128, 128)


In [ ]:
# =========================
# TRAIN TEST SPLIT
# =========================
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
# =========================
# TORCH DATA
# =========================
import torch

X_train = torch.tensor(X_train, dtype=torch.float32)
X_val   = torch.tensor(X_val, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_val   = torch.tensor(y_val, dtype=torch.long)

In [ ]:
# =========================
# DATALOADER
# =========================
from torch.utils.data import DataLoader, TensorDataset

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    TensorDataset(X_val, y_val),
    batch_size=16
)

In [ ]:
# =========================
# MODEL (MATCH TESTING)
# =========================
import torch.nn as nn

class AudioCNN(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(1,32,3,padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*16*16,256),  # fixed
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256,num_classes)
        )

    def forward(self,x):
        x = self.conv(x)
        x = self.fc(x)
        return x

In [ ]:
# =========================
# TRAIN
# =========================

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

from tqdm import tqdm

epochs = 25

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for xb, yb in tqdm(train_loader):
        xb, yb = xb.to(device), yb.to(device)

        out = model(xb)
        loss = criterion(out, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss:", total_loss/len(train_loader))

100%|██████████| 68/68 [00:01<00:00, 67.38it/s]


Epoch 1 Loss: 0.2597966120561871


100%|██████████| 68/68 [00:00<00:00, 70.07it/s]


Epoch 2 Loss: 0.25386693857851295


100%|██████████| 68/68 [00:00<00:00, 69.54it/s]


...

Epoch 23 Loss: 0.18159169390085428


100%|██████████| 68/68 [00:00<00:00, 68.37it/s]


Epoch 24 Loss: 0.21881390760844463


100%|██████████| 68/68 [00:00<00:00, 68.94it/s]

Epoch 25 Loss: 0.1999981333980994


In [ ]:
# =========================
# EVALUATION
# =========================
from sklearn.metrics import accuracy_score

model.eval()
preds, true = [], []

with torch.no_grad():
    for xb, yb in val_loader:
        xb = xb.to(device)

        out = model(xb)
        p = torch.argmax(out, dim=1)

        preds.extend(p.cpu().numpy())
        true.extend(yb.numpy())

print("Accuracy:", accuracy_score(true, preds))

Accuracy: 0.9628252788104089


In [ ]:
# =========================
# SAVE MODEL + LABEL MAP
# =========================
import json

torch.save(model.state_dict(), "audio_model.pth")

label_map = {i: label for i, label in enumerate(le.classes_)}

with open("audio_label_map.json", "w") as f:
    json.dump(label_map, f)

print("Saved!")

Saved!


In [ ]:
import json

label_map = {int(i): label for i, label in enumerate(le.classes_)}

with open("/content/audio_label_map.json", "w") as f:
    json.dump(label_map, f)

print("Label map saved!")

Label map saved!


In [ ]:
this ig good